## Today {.smaller}
The first half of today finished computer vision. This is the second half.

Everything so far assumed a **fixed-size grid** where **nearby things are
related**. Text has neither property.

- **Sequence data.** Tokens, embeddings, padding, masking. You will use this
  every week from here on.
- **1D convolution**, then **recurrent networks**: the two ways to process a
  sequence before attention existed.
- **Why that breaks.** Two measured failures, and they are why the field moved
  on.
- **LSTM**, briefly, as the patch that helped.

## Labs and quizzes {.smaller}

**Lab 6 was due yesterday.** That was transfer learning, then a pretrained
detector and a pretrained segmenter on the course photograph.

**Quiz 7 is at the end of class.** It covers the Oct 8 lecture: the back half
of the CNN deck through to non-maximum suppression. Nothing from today is on
it, and nothing past NMS is on it either. The study guide lists the scope
exactly.

**Lab 7 goes out today**, due Wednesday Oct 21. It is sequence data mechanics:
tokenizing text, building a vocabulary, padding a ragged batch and masking it
correctly. Deliberately not about LSTM architecture, because padding and
masking are what Weeks 9 and 10 need and the gates are not.

## What a convolution assumes {.smaller}
**A fixed-size input.** Every image is resized to $224 \times 224$ before it
goes in. You cannot resize a sentence.

**Locality.** A $3 \times 3$ kernel looks at a pixel and its neighbours,
because in an image the neighbours are the context. In text they often are not.

[![](images/fig-dependency.png){.dw95}](images/fig-dependency.png){target="_blank" .zoom}

# Sequence data

## What makes data sequential {.smaller}
**Sequence data is data where the order of the elements carries meaning and the
number of elements is not fixed.**

[![](images/seq-applications.png){.dw80}](images/seq-applications.png){target="_blank" .zoom}

Both halves of the definition matter. A bag of pixels in random order is no
longer an image, so images care about order too, but an image's size is fixed by
resizing and a sentence's is not. Notice that the input and the output are
different shapes in almost every row: one label from a sequence, a sequence from
one label, a sequence from a sequence.

## Tokenization {.smaller}

A network takes numbers. Text is characters. **Tokenization is the step that
splits text into pieces, and a vocabulary is the lookup from those pieces to
integer ids.**

[![](images/fig-tokenize.png){.dw95}](images/fig-tokenize.png){target="_blank" .zoom}

The same sentence is 37 tokens as characters, 8 as words, and
11 as subwords. The trade is between sequence length and vocabulary
size, and subword tokenization is what every modern model uses because it gets
most of both: a small fixed vocabulary that can still spell anything.

## Why subword tokenization won {.smaller}
:::: {.columns}
::: {.column width="33%"}
**Characters** spell anything with a few hundred tokens. The cost is long
sequences, and work grows with length.
:::
::: {.column width="33%"}
**Words** give short sequences and a vocabulary that never stops growing.
Anything missing becomes `<unk>`, meaning gone.
:::
::: {.column width="33%"}
**Subwords** keep frequent words whole and split rare ones. Fixed vocabulary,
nothing unspellable.
:::
::::

In the figure, `unhappier` became `un` + `happ` + `ier`. A model that has never
seen the word has seen `un` on `unlikely` and `ier` on `easier`, so the
morphology survives instead of collapsing to `<unk>`.

## One-hot encoding {.smaller}

The first thing you would try. **A one-hot vector is all zeros with a single 1,
at the index of the token.** A sequence of $s$ tokens over a vocabulary of $n$
becomes an $s \times n$ matrix $\mathbf{O}$, almost all of it zeros.

[![](images/onehot-to-embedding.png){.dw92}](images/onehot-to-embedding.png){target="_blank" .zoom}

Multiply $\mathbf{O}$ by a weight matrix $\mathbf{W_E}$ and you get a dense
$s \times m$ matrix $\mathbf{E}$, with $m$ in the hundreds rather than tens of
thousands. **That multiply is the embedding**, and it is the next slide.

## Embeddings {.smaller}
Multiplying a one-hot row by $\mathbf{W_E}$ just selects a row of
$\mathbf{W_E}$, so you may as well index it. **An embedding is that lookup
table: one learned row per token.**

[![](images/embedding-vectors.png){.dw85}](images/embedding-vectors.png){target="_blank" .zoom}

The rows are parameters, so the model learns them, and tokens that behave alike
end up near each other. One-hot cannot do that: every token sits exactly as far
from every other as it started, forever.

## How an embedding is read and updated {.smaller}

[![](images/fig-embed.png){.dw92}](images/fig-embed.png){target="_blank" .zoom}

## Embedding a batch of ids {.smaller .codetight}

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(6600)
vocab_size, d_model = 50_000, 256
emb = nn.Embedding(vocab_size, d_model)

ids = torch.randint(0, vocab_size, (4, 12))      # 4 sequences, 12 tokens each
print(f"ids      {tuple(ids.shape)}  dtype {ids.dtype}")
print(f"embedded {tuple(emb(ids).shape)}  dtype {emb(ids).dtype}")
print(f"parameters in the table: {emb.weight.numel():,}")
print("\nforward, the row IS the lookup, no matrix multiply:")
print(f"  emb(tensor(7)) == emb.weight[7]  ->  "
      f"{torch.equal(emb(torch.tensor(7)), emb.weight[7])}")

# backward: token 7 appears three times, token 3 once, nothing else appears
small = nn.Embedding(10, 4)
small(torch.tensor([[7, 3, 7], [7, 2, 2]])).sum().backward()
g = small.weight.grad.norm(dim=1)
print("\nbackward, gradient reaching each row:")
for row in (7, 3, 2, 5):
    print(f"  row {row}: {g[row]:.1f}", end="")
    print("   <- appeared 3 times" if row == 7 else
          ("   <- never appeared" if g[row] == 0 else ""))
print(f"  rows with any gradient at all: "
      f"{torch.nonzero(g).flatten().tolist()} of 10")

ids      (4, 12)  dtype torch.int64
embedded (4, 12, 256)  dtype torch.float32
parameters in the table: 12,800,000

forward, the row IS the lookup, no matrix multiply:
  emb(tensor(7)) == emb.weight[7]  ->  True

backward, gradient reaching each row:
  row 7: 6.0   <- appeared 3 times
  row 3: 2.0
  row 2: 4.0
  row 5: 0.0   <- never appeared
  rows with any gradient at all: [2, 3, 7] of 10


## Padding {.smaller}

A batch has to be a rectangle. Sentences are not the same length. **Padding
means appending a reserved token to every sequence in the batch until they all
reach the length of the longest one.**

[![](images/fig-padding.png){.dw92}](images/fig-padding.png){target="_blank" .zoom}

Four sentences, padded to 7, and 14 of the 28 cells are
padding. The pad id is reserved, conventionally 0, and it means "nothing here."

## Masking {.smaller}
**A mask is a boolean tensor, the same shape as the batch, marking which
positions are real.** Without one the model reads `PAD` as a word.

:::: {.columns}
::: {.column width="50%"}
The previous figure measured the cost: up to **86%** error in a pooled
vector, worst for the shortest sentence and zero for the one that needed no
padding. **A masking bug damages short examples and leaves long ones alone**, so
it hides in the average.
:::
::: {.column width="50%"}
Three places it must be applied, and forgetting any one is silent:

- **Pooling.** Divide by the real count, not $T$.
- **The loss.** `ignore_index=0`.
- **Attention**, next week. Pad positions go to $-\infty$ before the softmax.
:::
::::

## Pooling a padded batch {.smaller .codetight}

In [2]:
# One ragged batch, done correctly, in the six lines it actually takes.
from torch.nn.utils.rnn import pad_sequence

seqs = [torch.tensor([5, 9, 2]),
        torch.tensor([7, 1, 4, 8, 3, 6, 2]),
        torch.tensor([4]),
        torch.tensor([9, 9, 1])]

ids = pad_sequence(seqs, batch_first=True, padding_value=0)
mask = (ids != 0)

e = nn.Embedding(10, 16, padding_idx=0)(ids)
naive = e.mean(dim=1)                                        # wrong
pooled = (e * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)

print(f"padded ids   {tuple(ids.shape)}")
print(f"real tokens  {mask.sum(1).tolist()}   of {ids.shape[1]} columns")
print(f"pooled       {tuple(pooled.shape)}")
print()
err = (naive - pooled).norm(dim=1) / pooled.norm(dim=1) * 100
print("error from pooling without the mask, per sequence:")
for n, pct in zip(mask.sum(1).tolist(), err.tolist()):
    print(f"  length {n}:  {pct:5.1f}%")

padded ids   (4, 7)
real tokens  [3, 7, 1, 3]   of 7 columns
pooled       (4, 16)

error from pooling without the mask, per sequence:
  length 3:   57.1%
  length 7:    0.0%
  length 1:   85.7%
  length 3:   57.1%


## Building the batch {.smaller}
A `Dataset` returns one example; the default collation stacks them and needs
matching shapes. Ragged sequences do not have them, so you pass a
**`collate_fn`** and pad there, to each batch's own longest member.

[![](images/fig-bucketing.png){.dw92}](images/fig-bucketing.png){target="_blank" .zoom}

Sorting by length first is called **bucketing**. **For a time series, split on
time, not at random**: train on the past, validate on the future, or you have
leaked the answer.

## Collating a ragged batch {.smaller .codetight}

In [3]:
from torch.utils.data import DataLoader, Dataset

class Sentences(Dataset):
    def __init__(self, seqs, labels):
        self.seqs, self.labels = seqs, labels
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, i):
        return self.seqs[i], self.labels[i]

def collate(batch):
    """Pad to the longest member of THIS batch, and return the mask with it."""
    seqs, labels = zip(*batch)
    ids = pad_sequence(seqs, batch_first=True, padding_value=0)
    return ids, (ids != 0), torch.tensor(labels)

loader = DataLoader(Sentences(seqs, [0, 1, 1, 0]), batch_size=2,
                    shuffle=False, collate_fn=collate)

for ids, mask, y in loader:
    print(f"batch {tuple(ids.shape)}   real tokens {mask.sum(1).tolist()}   y {y.tolist()}")

batch (2, 7)   real tokens [3, 7]   y [0, 1]
batch (2, 3)   real tokens [1, 3]   y [1, 0]


# Recurrent networks

## 1D convolution {.smaller}
Slide a kernel along the sequence instead of across an image. `nn.Conv1d` takes
$(N, C, T)$ and accepts any $T$, so **variable length was never the real
objection**. Locality is.

[![](images/fig-conv1d.png){.dw95}](images/fig-conv1d.png){target="_blank" .zoom}

A $k$-wide kernel widens the receptive field by $k - 1$ per layer, measured
above by gradient tracing, so two tokens 100 apart need **50 layers**
where attention needs one.

**This is current practice, not history.** Speech models put a convolutional
front end before the transformer, Conformer blocks run convolution and attention
side by side, and dilated causal convolutions are a strong, cheap baseline for
time series. Use them where the pattern is local; reach for attention when it is
not.

## The recurrence idea {.smaller}
**A recurrent network keeps a hidden state $h_t$ that it updates at every step
from the current input and the previous state.**

:::: {.columns}
::: {.column width="50%"}
[![](images/unrolled-rnn.png){.dw95}](images/unrolled-rnn.png){target="_blank" .zoom}
:::
::: {.column width="50%"}
$$h_t = \tanh\!\left(W_x\,x_t + W_h\,h_{t-1} + b\right)$$

$h_0$ is zeros. **The same weights run at every step**, which is weight
sharing, the same idea as a kernel sliding over an image, except over time. It
is what lets one model take any length.
:::
::::

**The only route from `THE` to `DOG` is along the $W_h$ arrows**, eight
multiplications, with no other path.

## What an RNN layer returns {.smaller .codetight}

In [4]:
rnn = nn.RNN(input_size=16, hidden_size=32, batch_first=True)

x = torch.randn(4, 12, 16)              # 4 sequences, 12 steps, 16 features
out, h_T = rnn(x)

print(f"input          {tuple(x.shape)}")
print(f"out            {tuple(out.shape)}   the hidden state at EVERY step")
print(f"h_T            {tuple(h_T.shape)}   the hidden state at the LAST step")
print(f"and they agree: {torch.allclose(out[:, -1], h_T[0], atol=1e-6)}")
print()
print("the same weights at every step, so the count does not depend on T:")
for name, p in rnn.named_parameters():
    print(f"  {name:12s} {tuple(p.shape)}")

input          (4, 12, 16)
out            (4, 12, 32)   the hidden state at EVERY step
h_T            (1, 4, 32)   the hidden state at the LAST step
and they agree: True

the same weights at every step, so the count does not depend on T:
  weight_ih_l0 (32, 16)
  weight_hh_l0 (32, 32)
  bias_ih_l0   (32,)
  bias_hh_l0   (32,)


## Reading the output {.smaller}
A recurrent layer returns the hidden state at every step. Which part you read
is what makes it a different task.

[![](images/sequence-mappings.png){.dw85}](images/sequence-mappings.png){target="_blank" .zoom}

**Translation** is where attention came from: the whole input is squeezed into
**one** hidden state and the decoder works from that alone, whether the
sentence is 4 words or 400. With padding, `out[:, -1]` is the state after the
padding, not after the sentence.

## Generating one token at a time {.smaller}
[![](images/autoregressive-steps.png){.dw85}](images/autoregressive-steps.png){target="_blank" .zoom}

Predict one token, append it to the input, run again. **This is how every text
generation model you have used produces output**, one token per forward pass,
and attention does not change it.

Training does not work this way. Feeding the model its own early predictions
compounds the errors, so you feed it the true previous token instead. That is
**teacher forcing**, and it is why training is parallel over positions while
generation is not.

## Backpropagation through time {.smaller}
Unroll it: $T$ steps become $T$ copies of the same layer, an ordinary
feedforward network $T$ deep, and you backpropagate through that. The name
**backpropagation through time** is the only new part.

The weights are shared, so $W_h$ collects a gradient from every one of those
steps, and every step on the way back multiplies by the same Jacobian.

Week 6 hit this same wall with depth in CNNs: "the gradients just never make it
back." There the fix was a residual connection, a path with gradient 1 running
alongside the layers. **Recurrence has no such path, because the recurrence is
the path.**

## How far back the gradient reaches {.smaller}

[![](images/fig-decay.png){.dw95}](images/fig-decay.png){target="_blank" .zoom}

A loss at the final step, 20 random initializations, sequence length
120. Both lines are straight on a log axis, which means both decay
geometrically. The vanilla RNN keeps 0.53 of the gradient per step and
has nothing left after 20 steps; the LSTM keeps 0.62 and
lasts 28. Gating changed the slope. It did not change the shape.

## Why it is exponential {.smaller}
$$\frac{\partial h_T}{\partial h_t}
  = \prod_{k=t+1}^{T} \frac{\partial h_k}{\partial h_{k-1}}$$

[![](images/fig-bptt.png){.dw95}](images/fig-bptt.png){target="_blank" .zoom}

A product of $T - t$ terms, so it either goes to zero or it explodes. No choice
of weights makes a product of many terms behave like a sum. Exploding is the
easy half, because you can clip the norm; vanishing is the hard half, because
there is nothing to clip.

**The cost is not that the model is wrong about long sequences. It is that it
cannot learn anything that depends on one**, because the signal that would teach
it never arrives. Training longer does not help, and nor does a bigger hidden
state.

## One step per token {.smaller}
[![](images/fig-parallel.png){.dw95}](images/fig-parallel.png){target="_blank" .zoom}

$h_t$ needs $h_{t-1}$, so the steps cannot run together. The LSTM's cost per
token is flat: a fixed price per token, because it takes one sequential step
per token. At 1024 tokens that is 16 ms against 7.6 ms on
one RTX 3080.

**The win is parallelism, not less work.** Attention compares every position
with every other, so its total work grows with the *square* of the length. You
can watch the quadratic term arrive on the right, where its per-token cost
turns upward past 256. That is also why long context windows are expensive.

# Gated recurrence

## LSTM {.smaller}
An **LSTM** is a recurrent network with a second state that information travels
along without being multiplied by a weight matrix at every step.

:::: {.columns}
::: {.column width="42%"}
[![](images/lstm-cell.png){.dw95}](images/lstm-cell.png){target="_blank" .zoom}

[![](images/gate-symbols.png){.dw100}](images/gate-symbols.png){target="_blank" .zoom}
:::
::: {.column width="58%"}
The **cell state** $c_t$ is the track across the top, updated almost additively:

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

The circled regions are **gates**, each a learned function of $x_t$ and
$h_{t-1}$ squashed to $[0, 1]$, so each is a soft switch: what to **forget**,
what to **write**, what to **expose**.

When $f_t$ is near 1, $c_t \approx c_{t-1} + \text{something}$, so the
gradient along that track is multiplied by roughly 1 instead of a weight
matrix. **That is the residual connection's trick, eight years early.** A GRU
does the same with two gates.
:::
::::

## What gating fixed, and what it did not {.smaller}
:::: {.columns}
::: {.column width="50%"}
**Fixed.** An LSTM holds information far longer, and for a decade it was the
default for translation, speech and captioning. Measured reach 28
steps against 20, and more than that once the forget gate is
initialized well.
:::
::: {.column width="50%"}
**Not fixed.** The decay is still exponential, because the path is still a
path. And gating does nothing at all about the second failure: still one
sequential step per token.
:::
::::

Gating bought a decade without changing either of the two things that needed
changing. That is why this section is five minutes.

# Next week: attention

## The bottleneck attention removes {.smaller}
A recurrent encoder-decoder squeezes the whole input into one fixed-size vector,
so early positions must survive to the end and every step waits for the last.

[![](images/attention-matrix.png){.dw85}](images/attention-matrix.png){target="_blank" .zoom}

**Attention stops compressing.** Every position reads every other position
directly, in one hop, with weights computed from the content. Position 100
reaches position 1 without 99 multiplications, and since there is no order to
the computation it is one large matrix multiply. Hence *Attention Is All You
Need*, 2017: once you have this, the recurrence can go.

## What carries forward {.smaller}

Most of the first half of today is not about recurrent networks and does not
expire with them.

:::: {.columns}
::: {.column width="50%"}
**Used unchanged next week**

- Tokenization and the vocabulary
- Embeddings as a learned lookup table
- Padding a batch to its longest member
- Masking, including the attention mask
- `collate_fn` and bucketing by length
- Reading `(N, T, d)` shapes
:::
::: {.column width="50%"}
**Of historical interest**

- The recurrence $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$
- Backpropagation through time
- LSTM and GRU gates

You should recognize these, know what problem each was solving, and be able to
say why the field left them. You will not be asked to build one from scratch.
:::
::::

# Wrap-up

## Wrap-up {.smaller}
- **Sequence data** has variable length and meaningful order, the two
  properties convolution assumes away.
- **Subword tokenization** keeps a fixed vocabulary that can still spell
  anything.
- **Embeddings** are a learned lookup table, one row per token.
- **Padding** makes a batch rectangular; **masking** keeps it from being read
  as data, and forgetting it is silent.
- **1D convolution** handles any length but reaches only $k - 1$ further per
  layer. **Recurrent networks** share weights across time instead.
- **Two failures.** The gradient reaching back decays exponentially, so long
  dependencies cannot be learned, and each step waits for the previous one.
- **Gating** slowed the first and did nothing about the second.
- **Attention** fixes both, next week.

## Supplemental content {.smaller}

Optional. Prof. Hickman's
[centralized lecture content](https://jfh.georgetown.domains/centralized-lecture-content/)
covers this material with different examples, and a second pass rarely hurts:

- [Recurrent neural networks](https://jfh.georgetown.domains/centralized-lecture-content/content/deep-learning/rnn/)
- [Text preprocessing and embeddings](https://jfh.georgetown.domains/centralized-lecture-content/content/deep-learning/nlp/)
- [Attention and transformers](https://jfh.georgetown.domains/centralized-lecture-content/content/deep-learning/transformers/), which is next week

Also optional, and worth the time if you want the primary sources: Hochreiter
and Schmidhuber's LSTM paper (1997), Bahdanau et al. on attention for
translation (2014), and Vaswani et al., *Attention Is All You Need* (2017).

## Lab 7 and Quiz 7 {.smaller}

**Lab 7** is out today and due Wednesday Oct 21. Tokenize a corpus, build a
vocabulary, pad and mask a ragged batch, and pool it correctly, then confirm by
measurement that the unmasked version is wrong. Short, and entirely mechanics.

**Quiz 7** is now. Scope is the Oct 8 lecture only: the back half of the CNN
deck and detection as far as non-maximum suppression. A formula sheet comes
with it.

Next week is **Transformers**: self-attention, multi-head attention, positional
encoding, and the encoder-decoder architecture that the rest of the course is
built on.